# Invoice Extraction — Self-Contained Colab (Mimics main.py)

This notebook **reimplements** the invoice extraction pipeline **inline** (no imports from the project package). Same problem and solution:
- **Input:** Invoice images (e.g. `batch1-0348.jpg`).
- **Pipeline A:** OCR (Tesseract) → LLM (Gemini/OpenAI) → structured JSON.
- **Pipeline B:** OCR (Tesseract) → regex/heuristics → same fields.
- **Output:** Normalize → compare A vs B → reconcile (prefer B) → `output.csv` + `comparison_report.csv` + metrics.

## 1. Install dependencies

In [ ]:
!apt-get update -qq && apt-get install -y -qq tesseract-ocr
!pip install -q Pillow pytesseract openai google-generativeai pydantic python-dotenv

## 2. Config and constants

In [ ]:
import os
import re
import json
import csv
from pathlib import Path

# --- Paths (set INVOICE_IMAGES_DIR if you use Drive or another folder) ---
try:
    from google.colab import userdata
    os.environ.setdefault("GEMINI_API_KEY", userdata.get("GEMINI_API_KEY", ""))
except Exception:
    pass

IMAGES_DIR = Path(os.getenv("INVOICE_IMAGES_DIR", "invoice_extractor/images"))
OUTPUTS_DIR = Path(os.getenv("INVOICE_OUTPUTS_DIR", "invoice_extractor/outputs"))
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_PREFIX = os.getenv("INVOICE_IMAGE_PREFIX", "batch1-")
IMAGE_START = int(os.getenv("INVOICE_IMAGE_START", "348"))
IMAGE_END = int(os.getenv("INVOICE_IMAGE_END", "350"))
IMAGE_EXT = os.getenv("INVOICE_IMAGE_EXT", "jpg")

REQUIRED_FIELDS = [
    "seller_name", "seller_tax_id", "client_name", "client_tax_id",
    "invoice_number", "invoice_date", "net_worth", "vat", "gross_worth",
]

LLM_PROVIDER = os.getenv("INVOICE_LLM_PROVIDER", "gemini").lower()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
GEMINI_MODEL = os.getenv("GEMINI_INVOICE_MODEL", "gemini-1.5-flash")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL = os.getenv("OPENAI_INVOICE_MODEL", "gpt-4o-mini")

FLOAT_TOLERANCE = float(os.getenv("INVOICE_FLOAT_TOLERANCE", "0.01"))

def get_image_paths():
    paths = []
    for i in range(IMAGE_START, IMAGE_END + 1):
        name = f"{IMAGE_PREFIX}{i:04d}.{IMAGE_EXT}"
        p = IMAGES_DIR / name
        if p.exists():
            paths.append(p)
        else:
            print(f"Warning: image not found {p}")
    return paths

## 3. OCR + Pipeline A (LLM)

In [ ]:
import pytesseract
from PIL import Image

EXTRACTION_PROMPT = """You are an invoice data extractor. Extract the following fields from the invoice text below. Return ONLY valid JSON with exactly these keys (use null for missing values). No explanation, no markdown, no code block.

Required keys:
- seller_name (string)
- seller_tax_id (string)
- client_name (string)
- client_tax_id (string)
- invoice_number (string)
- invoice_date (string, any format)
- net_worth (number)
- vat (number)
- gross_worth (number)

Invoice text:
---
{text}
---

Return only the JSON object, nothing else."""

def _repair_json(s):
    s = s.strip()
    s = re.sub(r",\s*([}\]])", r"\1", s)
    open_braces = s.count("{") - s.count("}")
    open_brackets = s.count("[") - s.count("]")
    if open_braces > 0 or open_brackets > 0:
        s = s + "]" * open_brackets + "}" * open_braces
    return s

def _parse_llm_json(content):
    content = content.strip()
    if content.startswith("```"):
        lines = content.split("\n")
        content = "\n".join(l for l in lines if not l.strip().startswith("```")).strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        pass
    repaired = _repair_json(content)
    return json.loads(repaired)

def run_ocr(image_path):
    img = Image.open(image_path)
    if img.mode not in ("L", "RGB"):
        img = img.convert("RGB")
    return pytesseract.image_to_string(img) or ""

def extract_invoice_pipeline_a(image_path):
    try:
        text = run_ocr(image_path)
        if not text.strip():
            return {f: None for f in REQUIRED_FIELDS}
        prompt = EXTRACTION_PROMPT.format(text=text[:12000])
        if LLM_PROVIDER == "gemini":
            import google.generativeai as genai
            if not GEMINI_API_KEY:
                raise RuntimeError("GEMINI_API_KEY not set. Get one at https://aistudio.google.com/apikey")
            genai.configure(api_key=GEMINI_API_KEY)
            model = genai.GenerativeModel(GEMINI_MODEL)
            response = model.generate_content(prompt, generation_config=genai.types.GenerationConfig(temperature=0))
            content = (response.text or "").strip()
        else:
            from openai import OpenAI
            if not OPENAI_API_KEY:
                raise RuntimeError("OPENAI_API_KEY not set")
            client = OpenAI(api_key=OPENAI_API_KEY)
            r = client.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": prompt}], temperature=0)
            content = (r.choices[0].message.content or "").strip()
        data = _parse_llm_json(content)
        allowed = set(REQUIRED_FIELDS)
        return {k: data.get(k) for k in REQUIRED_FIELDS}
    except Exception as e:
        print(f"Pipeline A failed {image_path.name}: {e}")
        return None

## 4. Pipeline B (regex)

In [ ]:
_NAME_CAPTURE = r"[A-Za-z0-9 ,&.'\-]+"
PATTERNS = {
    "invoice_number": [r"Invoice\s*no[:\s]*([0-9]+)"],
    "invoice_date": [r"\b(\d{2}/\d{2}/\d{4})\b"],
    "seller_name": [r"Seller:\s*\n\s*(" + _NAME_CAPTURE + r")", r"Seller:\s*(" + _NAME_CAPTURE + r")"],
    "seller_tax_id": [r"Tax\s*Id[:\s]*([0-9\-]+)"],
    "client_name": [r"Client:\s*\n\s*(" + _NAME_CAPTURE + r")", r"Client:\s*(" + _NAME_CAPTURE + r")"],
    "client_tax_id": [r"Tax\s*Id[:\s]*([0-9\-]+)"],
    "net_worth": [r"Net worth\s*\n\s*[\$]?\s*([0-9 ,\.]+)"],
    "vat": [r"VAT\s*\n\s*[\$]?\s*([0-9 ,\.]+)"],
    "gross_worth": [r"Gross worth\s*\n\s*[\$]?\s*([0-9 ,\.]+)"],
}

def _first_match(text, patterns):
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return m.group(1).strip() if m.lastindex else None
    return None

def _parse_number(s):
    if not s:
        return None
    cleaned = re.sub(r"[^\d.,\-]", "", s.replace(" ", "")).replace(",", ".")
    if not cleaned:
        return None
    try:
        return float(cleaned)
    except ValueError:
        return None

def _extract_names_heuristic(text):
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    seller = client = None
    for i, line in enumerate(lines):
        if re.search(r"(sold\s+to|client|buyer|customer)\s*[:\s]*", line, re.I):
            rest = re.sub(r"^(?:sold\s+to|client|buyer|customer)\s*[:\s]*", "", line, flags=re.I).strip()
            if rest and len(rest) > 2:
                client = rest
            elif i + 1 < len(lines) and len(lines[i + 1]) > 2:
                client = lines[i + 1]
            break
    for line in lines[:15]:
        if len(line) < 4 or re.match(r"^[\d\s.,\-/]+$", line) or re.search(r"invoice|date|vat|tax|net|gross", line, re.I):
            continue
        if not seller:
            seller = line
            break
    return seller, client

def extract_structured(text):
    result = {f: None for f in REQUIRED_FIELDS}
    text_nl = text.replace("\r", "\n")
    seller_name = _first_match(text_nl, PATTERNS["seller_name"])
    client_name = _first_match(text_nl, PATTERNS["client_name"])
    if seller_name is None or client_name is None:
        heur_s, heur_c = _extract_names_heuristic(text_nl)
        if seller_name is None: seller_name = heur_s
        if client_name is None: client_name = heur_c
    result["seller_name"] = seller_name
    result["client_name"] = client_name
    result["seller_tax_id"] = _first_match(text_nl, PATTERNS["seller_tax_id"])
    result["client_tax_id"] = _first_match(text_nl, PATTERNS["client_tax_id"])
    result["invoice_number"] = _first_match(text_nl, PATTERNS["invoice_number"])
    result["invoice_date"] = _first_match(text_nl, PATTERNS["invoice_date"])
    result["net_worth"] = _parse_number(_first_match(text_nl, PATTERNS["net_worth"]))
    result["vat"] = _parse_number(_first_match(text_nl, PATTERNS["vat"]))
    result["gross_worth"] = _parse_number(_first_match(text_nl, PATTERNS["gross_worth"]))
    return result

def extract_invoice_pipeline_b(image_path):
    try:
        text = run_ocr(image_path)
        if not text.strip():
            return {f: None for f in REQUIRED_FIELDS}
        return extract_structured(text)
    except Exception as e:
        print(f"Pipeline B failed {image_path.name}: {e}")
        return None

## 5. Normalizer and validator

In [ ]:
def normalize_currency(value):
    if value is None: return None
    if isinstance(value, (int, float)): return float(value) if value == value else None
    s = str(value).strip()
    if not s or s.lower() in ("", "n/a", "nan", "null", "-"): return None
    cleaned = re.sub(r"[^\d.\-]", "", s.replace(",", "."))
    parts = cleaned.split(".")
    if len(parts) > 2: cleaned = "".join(parts[:-1]) + "." + parts[-1]
    try: return float(cleaned)
    except ValueError: return None

def normalize_date(value):
    if value is None: return None
    s = str(value).strip()
    if not s or s.lower() in ("", "n/a", "nan", "null"): return None
    m = re.match(r"(\d{4})-(\d{2})-(\d{2})", s)
    if m:
        y, mo, d = m.groups()
        if 1 <= int(mo) <= 12 and 1 <= int(d) <= 31: return f"{y}-{mo}-{d}"
    for sep in [".", "/", "-"]:
        parts = re.split(r"[\s./\-]+", s)
        if len(parts) >= 3:
            try:
                d, mo, y = parts[0], parts[1], parts[2]
                y = int(y); mo = int(mo); d = int(d)
                if y < 100: y += 2000 if y < 50 else 1900
                if 1 <= mo <= 12 and 1 <= d <= 31: return f"{y:04d}-{mo:02d}-{d:02d}"
            except (ValueError, IndexError): continue
    return None

def normalize_tax_id(value):
    if value is None: return None
    s = str(value).strip(); return re.sub(r"\s+", "", s) if s else None

def normalize_name(value):
    if value is None: return None
    s = str(value).strip(); return " ".join(s.lower().split()) if s else None

def normalize_string(value):
    if value is None: return None
    s = str(value).strip(); return " ".join(s.split()) if s else None

CURRENCY_FIELDS = {"net_worth", "vat", "gross_worth"}
DATE_FIELDS = {"invoice_date"}
TAX_ID_FIELDS = {"seller_tax_id", "client_tax_id"}
NAME_FIELDS = {"seller_name", "client_name"}

def normalize_field(field_name, value):
    if value is None or (isinstance(value, str) and not value.strip()): return None
    if field_name in CURRENCY_FIELDS: return normalize_currency(value)
    if field_name in DATE_FIELDS: return normalize_date(value)
    if field_name in TAX_ID_FIELDS: return normalize_tax_id(value)
    if field_name in NAME_FIELDS: return normalize_name(value)
    return normalize_string(value)

def normalize_invoice_dict(data):
    return {k: normalize_field(k, data.get(k)) for k in REQUIRED_FIELDS}

def _values_match(field_name, a, b):
    if a is None and b is None: return True
    if a is None or b is None: return False
    if field_name in ("net_worth", "vat", "gross_worth"):
        try: return abs(float(a) - float(b)) <= FLOAT_TOLERANCE
        except (TypeError, ValueError): return str(a).strip() == str(b).strip()
    return str(a).strip().lower() == str(b).strip().lower()

def compare_invoices(file_name, pipeline_a_output, pipeline_b_output):
    rows = []
    for field_name in REQUIRED_FIELDS:
        va = pipeline_a_output.get(field_name)
        vb = pipeline_b_output.get(field_name)
        match = _values_match(field_name, va, vb)
        rows.append({"file_name": file_name, "field_name": field_name, "pipeline_a_value": va if va is None else str(va), "pipeline_b_value": vb if vb is None else str(vb), "match": match})
    return rows

def reconcile_value(field_name, pipeline_a_value, pipeline_b_value, prefer_b_on_mismatch=True):
    if _values_match(field_name, pipeline_a_value, pipeline_b_value):
        return pipeline_b_value if pipeline_b_value is not None else pipeline_a_value
    return pipeline_b_value if prefer_b_on_mismatch else pipeline_a_value

def build_reconciled_invoice(file_name, pipeline_a_output, pipeline_b_output):
    row = {"file_name": file_name}
    for field_name in REQUIRED_FIELDS:
        va = pipeline_a_output.get(field_name)
        vb = pipeline_b_output.get(field_name)
        row[field_name] = reconcile_value(field_name, va, vb, prefer_b_on_mismatch=True)
    return row

## 6. Main loop: process images, write CSVs, print metrics

In [ ]:
image_paths = get_image_paths()
if not image_paths:
    raise SystemExit(f"No images in {IMAGES_DIR} (prefix={IMAGE_PREFIX}, range={IMAGE_START}-{IMAGE_END})")

print(f"Found {len(image_paths)} images. Starting Pipeline A + B.")
output_rows = []
comparison_rows = []
total_fields = matched_fields = docs_processed = docs_full_match = 0

for path in image_paths:
    file_name = path.name
    print(f"Processing {file_name}")
    a_raw = extract_invoice_pipeline_a(path)
    b_raw = extract_invoice_pipeline_b(path)
    if a_raw is None: a_raw = {f: None for f in REQUIRED_FIELDS}
    if b_raw is None: b_raw = {f: None for f in REQUIRED_FIELDS}
    a_norm = normalize_invoice_dict(a_raw)
    b_norm = normalize_invoice_dict(b_raw)
    comp = compare_invoices(file_name, a_norm, b_norm)
    comparison_rows.extend(comp)
    reconciled = build_reconciled_invoice(file_name, a_norm, b_norm)
    output_rows.append(reconciled)
    doc_matches = sum(1 for r in comp if r["match"])
    total_fields += len(REQUIRED_FIELDS)
    matched_fields += doc_matches
    docs_processed += 1
    if doc_matches == len(REQUIRED_FIELDS): docs_full_match += 1
    for r in comp:
        if not r["match"]:
            print(f"  Mismatch {r['field_name']}: A={r['pipeline_a_value']} B={r['pipeline_b_value']}")

out_cols = ["file_name"] + REQUIRED_FIELDS
with open(OUTPUTS_DIR / "output.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=out_cols, extrasaction="ignore")
    w.writeheader()
    w.writerows(output_rows)
print(f"Wrote {OUTPUTS_DIR / 'output.csv'}")

report_cols = ["file_name", "field_name", "pipeline_a_value", "pipeline_b_value", "match"]
with open(OUTPUTS_DIR / "comparison_report.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=report_cols)
    w.writeheader()
    w.writerows(comparison_rows)
print(f"Wrote {OUTPUTS_DIR / 'comparison_report.csv'}")

field_accuracy = (matched_fields / total_fields * 100) if total_fields else 0
doc_accuracy = (docs_full_match / docs_processed * 100) if docs_processed else 0
print("\n--- Summary metrics ---")
print(f"Images processed:     {docs_processed}")
print(f"Field Accuracy:       {field_accuracy:.1f}%")
print(f"Document Accuracy:    {doc_accuracy:.1f}%")
print(f"Total field matches:  {matched_fields} / {total_fields}")
print(f"Documents full match: {docs_full_match} / {docs_processed}")
print("-----------------------")

## 7. (Optional) Download outputs

In [ ]:
from google.colab import files
print("Outputs:", list(OUTPUTS_DIR.iterdir()))
# files.download(str(OUTPUTS_DIR / "output.csv"))
# files.download(str(OUTPUTS_DIR / "comparison_report.csv"))